# HW02 — MLflow Experiment Tracking

This cleaned notebook trains and tracks reproducible binary-classification experiments for the Airbnb listing availability dataset.

It intentionally avoids hard-coded MLflow credentials. Put credentials in a local `.env` file or export them as environment variables before running.


## Required output

- At least five MLflow runs are logged.
- One intentionally leaky model is marked as invalid.
- Clean baseline, logistic-regression, threshold-tuned, and tree-based models are tracked.
- Dataset hash/version, parameters, metrics, model artifacts, and supporting artifacts are logged.
- A final clean candidate run is selected and tagged.


In [ ]:
import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv

import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

RANDOM_STATE = 42


## 1. Configure MLflow


In [ ]:
load_dotenv()

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://185.50.38.163:33014")
MLFLOW_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME")
STUDENT_NAME = os.getenv("STUDENT_NAME", "sobhan_jabari")

required_envs = {
    "MLFLOW_TRACKING_USERNAME": MLFLOW_USERNAME,
    "MLFLOW_TRACKING_PASSWORD": MLFLOW_PASSWORD,
    "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME,
}

missing = [key for key, value in required_envs.items() if not value]
if missing:
    raise ValueError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Create a local .env file from .env.example or export them before running."
    )

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)


## 2. Load the HW01 feature dataset


In [ ]:
candidate_paths = [
    Path("listing_availability_features_v1_student.parquet"),
    Path("listing_availability_features_v1_student.csv"),
    Path("data/features/listing_availability_features_v1_student.parquet"),
    Path("data/features/listing_availability_features_v1_student.csv"),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find the HW01 feature dataset in the project root or data/features/.")

if DATA_PATH.suffix == ".parquet":
    df = pd.read_parquet(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.dtypes)


## 3. Define target, leakage columns, and dataset metadata


In [ ]:
TARGET = "high_demand_proxy"

LEAKY_COLS = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
]

META_COLS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
]

if TARGET not in df.columns:
    raise KeyError(f"Target column {TARGET!r} is missing from the dataset.")

def get_dataframe_hash(frame: pd.DataFrame) -> str:
    hashed = pd.util.hash_pandas_object(frame, index=True).values
    return hashlib.md5(hashed).hexdigest()[:12]

DATASET_HASH = get_dataframe_hash(df)
DATASET_VERSION = df["dataset_version"].iloc[0] if "dataset_version" in df.columns else "unknown"

print("Target distribution:")
display(df[TARGET].value_counts(normalize=True).sort_index())
print("Dataset hash:", DATASET_HASH)
print("Dataset version:", DATASET_VERSION)
print("Leaky columns:", [c for c in LEAKY_COLS if c in df.columns])


## 4. Feature engineering


In [ ]:
def make_features(raw_df: pd.DataFrame, drop_leaky: bool = True):
    """Create model-ready features with optional future-leakage removal."""
    data = raw_df.copy()

    # Normalize common boolean-like columns.
    for col in ["instant_bookable", "is_superhost"]:
        if col in data.columns:
            data[col] = (
                data[col]
                .astype(str)
                .str.lower()
                .map({"t": 1, "true": 1, "1": 1, "yes": 1, "f": 0, "false": 0, "0": 0, "no": 0})
                .fillna(0)
                .astype(int)
            )

    drop_cols = [TARGET] + META_COLS
    if drop_leaky:
        drop_cols += LEAKY_COLS
    drop_cols = [col for col in drop_cols if col in data.columns]

    y = data[TARGET].astype(int)
    X = data.drop(columns=drop_cols)

    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    return X, y


X_clean, y_clean = make_features(df, drop_leaky=True)
X_leaky, y_leaky = make_features(df, drop_leaky=False)

clean_feature_cols = X_clean.columns.tolist()
leaky_feature_cols = X_leaky.columns.tolist()

print("Clean feature shape:", X_clean.shape)
print("Leaky feature shape:", X_leaky.shape)
print("Target shape:", y_clean.shape)
print("Leaky future features present in clean data:", sorted(set(LEAKY_COLS).intersection(X_clean.columns)))


## 5. Train/test split


In [ ]:
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean,
    y_clean,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_clean,
)

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_leaky,
)

print("Clean train/test:", X_train_clean.shape, X_test_clean.shape)
print("Leaky train/test:", X_train_leaky.shape, X_test_leaky.shape)
print("Train target rate:", y_train_clean.mean())
print("Test target rate:", y_test_clean.mean())


## 6. MLflow helpers


In [ ]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def safe_auc(y_true, y_score):
    """Compute AUC, returning NaN if it cannot be computed."""
    try:
        return roc_auc_score(y_true, y_score)
    except ValueError:
        return float("nan")


def evaluate_binary_classifier(model, X, y, threshold=0.5):
    y_score = get_positive_scores(model, X)
    y_pred = (y_score >= threshold).astype(int)
    metrics = {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "auc": safe_auc(y, y_score),
    }
    return metrics, y_pred, y_score


def log_confusion_matrix_artifact(y_true, y_pred, filename="confusion_matrix.txt"):
    cm = confusion_matrix(y_true, y_pred)
    with open(filename, "w", encoding="utf-8") as f:
        f.write("Confusion Matrix\n")
        f.write(str(cm))
        f.write("\n\nClassification Report\n")
        f.write(classification_report(y_true, y_pred, zero_division=0))
    mlflow.log_artifact(filename)


def log_sklearn_model_compatible(model, X_sample, artifact_path="model"):
    """Log a sklearn model with a signature when possible."""
    try:
        signature = infer_signature(X_sample, model.predict(X_sample))
        mlflow.sklearn.log_model(model, artifact_path=artifact_path, signature=signature)
    except Exception as exc:
        print("Model signature logging failed; logging model without signature:", exc)
        mlflow.sklearn.log_model(model, artifact_path=artifact_path)


def run_experiment(
    run_name,
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    params,
    leaky,
    description,
    threshold=0.5,
):
    """Fit, evaluate, and log one MLflow experiment."""
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("dataset_path", str(DATA_PATH))
        mlflow.log_param("dataset_hash", DATASET_HASH)
        mlflow.log_param("dataset_version", DATASET_VERSION)
        mlflow.log_param("n_rows", df.shape[0])
        mlflow.log_param("n_raw_columns", df.shape[1])
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("target", TARGET)
        mlflow.log_param("leaky_features_used", leaky)
        mlflow.log_param("threshold", threshold)
        mlflow.log_params(params)

        mlflow.set_tag("description", description)
        mlflow.set_tag("problem_type", "binary_classification")
        mlflow.set_tag("student", STUDENT_NAME)
        mlflow.set_tag("valid_model", "false" if leaky else "true")
        if leaky:
            mlflow.set_tag("reason", "uses future availability columns; leakage")

        model.fit(X_train, y_train)

        train_metrics, y_pred_train, y_score_train = evaluate_binary_classifier(model, X_train, y_train, threshold)
        test_metrics, y_pred_test, y_score_test = evaluate_binary_classifier(model, X_test, y_test, threshold)

        metrics = {f"train_{k}": v for k, v in train_metrics.items()}
        metrics.update({f"test_{k}": v for k, v in test_metrics.items()})
        metrics["auc_gap"] = metrics["train_auc"] - metrics["test_auc"]
        metrics["f1_gap"] = metrics["train_f1"] - metrics["test_f1"]
        mlflow.log_metrics(metrics)

        log_confusion_matrix_artifact(y_test, y_pred_test)

        feature_file = "feature_columns.json"
        with open(feature_file, "w", encoding="utf-8") as f:
            json.dump(list(X_train.columns), f, indent=2)
        mlflow.log_artifact(feature_file)

        metadata_file = "dataset_metadata_snapshot.json"
        with open(metadata_file, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "dataset_path": str(DATA_PATH),
                    "dataset_hash": DATASET_HASH,
                    "dataset_version": str(DATASET_VERSION),
                    "n_rows": int(df.shape[0]),
                    "n_raw_columns": int(df.shape[1]),
                    "target": TARGET,
                },
                f,
                indent=2,
            )
        mlflow.log_artifact(metadata_file)

        fitted_estimator = model.named_steps.get("model", model) if hasattr(model, "named_steps") else model
        if hasattr(fitted_estimator, "feature_importances_"):
            importance_df = pd.DataFrame(
                {"feature": X_train.columns, "importance": fitted_estimator.feature_importances_}
            ).sort_values("importance", ascending=False)
            importance_df.to_csv("feature_importance.csv", index=False)
            mlflow.log_artifact("feature_importance.csv")
            display(importance_df.head(10))

        log_sklearn_model_compatible(model, X_train, artifact_path="model")

        print("=" * 80)
        print(run_name)
        print("=" * 80)
        for key, value in metrics.items():
            print(f"{key}: {value:.4f}")

        return metrics, model


## 7. Run 0 — intentionally leaky model


In [ ]:
run0_leaky_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]
)

metrics_run0, fitted_run0_model = run_experiment(
    run_name="run0_leaky_logistic_regression_invalid",
    model=run0_leaky_model,
    X_train=X_train_leaky,
    X_test=X_test_leaky,
    y_train=y_train_leaky,
    y_test=y_test_leaky,
    params={
        "run_number": 0,
        "model_type": "LogisticRegression",
        "class_weight": "balanced",
        "max_iter": 2000,
        "split": "random_stratified",
        "uses_future_features": True,
    },
    leaky=True,
    description="Run 0: intentionally invalid model using future availability features.",
)


## 8. Run 1 — dummy baseline


In [ ]:
run1_dummy_model = DummyClassifier(strategy="most_frequent")

metrics_run1, fitted_run1_model = run_experiment(
    run_name="run1_dummy_baseline_clean",
    model=run1_dummy_model,
    X_train=X_train_clean,
    X_test=X_test_clean,
    y_train=y_train_clean,
    y_test=y_test_clean,
    params={
        "run_number": 1,
        "model_type": "DummyClassifier",
        "strategy": "most_frequent",
        "split": "random_stratified",
        "uses_future_features": False,
    },
    leaky=False,
    description="Run 1: clean dummy baseline using the majority class.",
)


## 9. Run 2 — clean logistic regression


In [ ]:
run2_logreg_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)

metrics_run2, fitted_run2_model = run_experiment(
    run_name="run2_logistic_regression_clean",
    model=run2_logreg_model,
    X_train=X_train_clean,
    X_test=X_test_clean,
    y_train=y_train_clean,
    y_test=y_test_clean,
    params={
        "run_number": 2,
        "model_type": "LogisticRegression",
        "class_weight": "none",
        "max_iter": 2000,
        "split": "random_stratified",
        "uses_future_features": False,
    },
    leaky=False,
    description="Run 2: clean logistic regression without future leakage features.",
)


## 10. Run 3 — class-weighted logistic regression


In [ ]:
run3_logreg_balanced = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]
)

metrics_run3, fitted_run3_model = run_experiment(
    run_name="run3_class_weighted_logistic_regression_clean",
    model=run3_logreg_balanced,
    X_train=X_train_clean,
    X_test=X_test_clean,
    y_train=y_train_clean,
    y_test=y_test_clean,
    params={
        "run_number": 3,
        "model_type": "LogisticRegression",
        "class_weight": "balanced",
        "max_iter": 2000,
        "split": "random_stratified",
        "uses_future_features": False,
    },
    leaky=False,
    description="Run 3: class-weighted logistic regression without future leakage features.",
)


## 11. Run 4 — threshold tuning


In [ ]:
y_proba_test_run3 = fitted_run3_model.predict_proba(X_test_clean)[:, 1]

threshold_results = []
for threshold in np.arange(0.10, 0.91, 0.01):
    y_pred_thr = (y_proba_test_run3 >= threshold).astype(int)
    threshold_results.append(
        {
            "threshold": float(threshold),
            "accuracy": accuracy_score(y_test_clean, y_pred_thr),
            "precision": precision_score(y_test_clean, y_pred_thr, zero_division=0),
            "recall": recall_score(y_test_clean, y_pred_thr, zero_division=0),
            "f1": f1_score(y_test_clean, y_pred_thr, zero_division=0),
        }
    )

threshold_df = pd.DataFrame(threshold_results)
best_threshold_row = threshold_df.sort_values("f1", ascending=False).iloc[0]
BEST_THRESHOLD = float(best_threshold_row["threshold"])
BEST_THRESHOLD_F1 = float(best_threshold_row["f1"])

print("Best threshold:", BEST_THRESHOLD)
print("Best F1:", BEST_THRESHOLD_F1)
display(threshold_df.sort_values("f1", ascending=False).head(10))


In [ ]:
with mlflow.start_run(run_name="run4_threshold_tuning_logistic_regression_clean"):
    y_proba_train = fitted_run3_model.predict_proba(X_train_clean)[:, 1]
    y_proba_test = fitted_run3_model.predict_proba(X_test_clean)[:, 1]

    y_pred_train_thr = (y_proba_train >= BEST_THRESHOLD).astype(int)
    y_pred_test_thr = (y_proba_test >= BEST_THRESHOLD).astype(int)

    metrics_run4 = {
        "train_accuracy": accuracy_score(y_train_clean, y_pred_train_thr),
        "test_accuracy": accuracy_score(y_test_clean, y_pred_test_thr),
        "train_precision": precision_score(y_train_clean, y_pred_train_thr, zero_division=0),
        "test_precision": precision_score(y_test_clean, y_pred_test_thr, zero_division=0),
        "train_recall": recall_score(y_train_clean, y_pred_train_thr, zero_division=0),
        "test_recall": recall_score(y_test_clean, y_pred_test_thr, zero_division=0),
        "train_f1": f1_score(y_train_clean, y_pred_train_thr, zero_division=0),
        "test_f1": f1_score(y_test_clean, y_pred_test_thr, zero_division=0),
        "train_auc": roc_auc_score(y_train_clean, y_proba_train),
        "test_auc": roc_auc_score(y_test_clean, y_proba_test),
    }
    metrics_run4["auc_gap"] = metrics_run4["train_auc"] - metrics_run4["test_auc"]
    metrics_run4["f1_gap"] = metrics_run4["train_f1"] - metrics_run4["test_f1"]

    mlflow.log_param("dataset_path", str(DATA_PATH))
    mlflow.log_param("dataset_hash", DATASET_HASH)
    mlflow.log_param("dataset_version", DATASET_VERSION)
    mlflow.log_param("n_rows", df.shape[0])
    mlflow.log_param("n_raw_columns", df.shape[1])
    mlflow.log_param("n_features", X_train_clean.shape[1])
    mlflow.log_param("target", TARGET)
    mlflow.log_param("leaky_features_used", False)
    mlflow.log_param("run_number", 4)
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("base_model", "run3_class_weighted_logistic_regression_clean")
    mlflow.log_param("threshold", BEST_THRESHOLD)
    mlflow.log_param("threshold_selection_metric", "test_f1")
    mlflow.log_param("uses_future_features", False)

    mlflow.set_tag("description", "Run 4: threshold tuning on class-weighted logistic regression.")
    mlflow.set_tag("problem_type", "binary_classification")
    mlflow.set_tag("student", STUDENT_NAME)
    mlflow.set_tag("valid_model", "true")

    mlflow.log_metrics(metrics_run4)

    threshold_df.to_csv("threshold_tuning_results.csv", index=False)
    mlflow.log_artifact("threshold_tuning_results.csv")

    log_confusion_matrix_artifact(
        y_test_clean,
        y_pred_test_thr,
        filename="confusion_matrix_threshold_tuned.txt",
    )

    log_sklearn_model_compatible(fitted_run3_model, X_train_clean, artifact_path="model")

    print("=" * 80)
    print("run4_threshold_tuning_logistic_regression_clean")
    print("=" * 80)
    for key, value in metrics_run4.items():
        print(f"{key}: {value:.4f}")


## 12. Run 5 — tree-based model


In [ ]:
run5_tree_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

metrics_run5, fitted_run5_model = run_experiment(
    run_name="run5_tree_based_random_forest_clean",
    model=run5_tree_model,
    X_train=X_train_clean,
    X_test=X_test_clean,
    y_train=y_train_clean,
    y_test=y_test_clean,
    params={
        "run_number": 5,
        "model_type": "RandomForestClassifier",
        "n_estimators": 300,
        "max_depth": 12,
        "min_samples_leaf": 10,
        "class_weight": "balanced",
        "split": "random_stratified",
        "uses_future_features": False,
    },
    leaky=False,
    description="Run 5: clean tree-based Random Forest model without future leakage features.",
)


## 13. Compare MLflow runs


In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_auc DESC"],
)

cols = [
    "run_id",
    "tags.mlflow.runName",
    "tags.valid_model",
    "metrics.test_auc",
    "metrics.test_f1",
    "metrics.test_accuracy",
    "metrics.test_precision",
    "metrics.test_recall",
    "metrics.train_auc",
    "metrics.auc_gap",
    "params.run_number",
    "params.model_type",
    "params.threshold",
    "params.leaky_features_used",
]

available_cols = [col for col in cols if col in runs.columns]
comparison_df = runs[available_cols].copy()
display(comparison_df)


## 14. Select final candidate


In [ ]:
valid_runs = runs[runs["tags.valid_model"] == "true"].copy()

valid_cols = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.test_auc",
    "metrics.test_f1",
    "metrics.test_accuracy",
    "metrics.test_precision",
    "metrics.test_recall",
    "metrics.auc_gap",
    "params.run_number",
    "params.model_type",
    "params.threshold",
]
valid_cols = [col for col in valid_cols if col in valid_runs.columns]

valid_comparison = valid_runs[valid_cols].sort_values(
    by=["metrics.test_auc", "metrics.test_f1"],
    ascending=False,
)
display(valid_comparison)

best_valid_run = valid_comparison.iloc[0]
BEST_RUN_ID = best_valid_run["run_id"]
BEST_RUN_NAME = best_valid_run["tags.mlflow.runName"]
BEST_TEST_AUC = best_valid_run["metrics.test_auc"]
BEST_TEST_F1 = best_valid_run["metrics.test_f1"]

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run ID:", BEST_RUN_ID)
print("Selected best run name:", BEST_RUN_NAME)
print("Best test AUC:", BEST_TEST_AUC)
print("Best test F1:", BEST_TEST_F1)


## Final explanation


In [ ]:
final_explanation = f'''
I selected {BEST_RUN_NAME} as the final candidate because it is a valid clean run that does not use future availability leakage columns and ranked highest among valid runs by test AUC, with test F1 used as a secondary comparison metric.

The intentionally leaky run was rejected even if it performs well because it uses future-calendar information that would not be available at prediction time. This would make the model unrealistically optimistic and unsuitable for real deployment.

The dummy baseline provides a minimum reference point, the logistic-regression runs show interpretable linear baselines, threshold tuning improves the precision/recall trade-off, and the random-forest run tests whether non-linear feature interactions improve performance.

As a next step, I would validate the selected model with a time-based split and run an additional stricter experiment that removes historical availability proxy features to test whether the model is overly dependent on availability-derived signals.
'''

print(final_explanation)
